# Training Data Generation with Fully Populated Graphs

This notebook demonstrates how to generate training data with fully populated graphs for GNN scheduling, where each sample contains execution times for all possible client-server pairs. We'll explore both single-task and batch processing approaches for creating bipartite graphs.


In [3]:
import sys
sys.path.append('../..')

import json
import pickle
from pathlib import Path
from typing import Dict, List, Tuple, Any
import random
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("Imports successful!")


Imports successful!


## 1. Key Innovation: Fully Populated Training Data

The main innovation of this approach is generating **complete execution time data** for all possible client-server-task combinations, rather than just the single execution that actually occurred. This provides much richer training data for GNNs.


In [4]:
# Define infrastructure topology
client_nodes = ["client_node0", "client_node1", "client_node2"]
server_nodes = ["server_node0", "server_node1", "server_node2", "server_node3"]

# Static conditions for consistent training data
static_conditions = {
    "queue_lengths": {0: 2, 1: 5, 2: 0, 3: 3, 4: 1, 5: 4},
    "memory_usage": {
        "client_node0": 0.3, "client_node1": 0.5, "client_node2": 0.2,
        "server_node0": 0.7, "server_node1": 0.4, "server_node2": 0.6, "server_node3": 0.8
    },
    "network_latencies": {}
}

# Generate network latencies for all client-server pairs
for client in client_nodes:
    for server in server_nodes:
        if client != server:
            if client == "client_node0":
                latency = random.uniform(0.001, 0.02)  # Good connectivity
            elif client == "client_node1":
                latency = random.uniform(0.02, 0.05)   # Medium connectivity
            else:
                latency = random.uniform(0.05, 0.1)    # Poor connectivity
            static_conditions["network_latencies"][(client, server)] = latency

print(f"Client nodes: {client_nodes}")
print(f"Server nodes: {server_nodes}")
print(f"Total client-server pairs: {len(client_nodes) * len(server_nodes)}")
print(f"Network latencies defined: {len(static_conditions['network_latencies'])}")


Client nodes: ['client_node0', 'client_node1', 'client_node2']
Server nodes: ['server_node0', 'server_node1', 'server_node2', 'server_node3']
Total client-server pairs: 12
Network latencies defined: 12


## 2. Calculate Execution Times for ALL Client-Server Pairs

This is the key innovation: instead of getting just one execution time per task, we calculate execution times for ALL possible client-server combinations.


In [5]:
# Define task types with realistic characteristics
task_types = {
    "dnn1": {
        "name": "DNN1",
        "platforms": ["rpiCpu", "xavierCpu", "xavierGpu"],
        "executionTime": {"rpiCpu": 2.0, "xavierCpu": 1.0, "xavierGpu": 0.5},
        "coldStartDuration": {"rpiCpu": 1.0, "xavierCpu": 0.5, "xavierGpu": 0.2},
        "memoryRequirements": {"rpiCpu": 0.5, "xavierCpu": 0.3, "xavierGpu": 0.2}
    },
    "dnn2": {
        "name": "DNN2", 
        "platforms": ["rpiCpu", "xavierCpu", "xavierGpu", "pynqFpga"],
        "executionTime": {"rpiCpu": 3.0, "xavierCpu": 1.5, "xavierGpu": 0.8, "pynqFpga": 1.2},
        "coldStartDuration": {"rpiCpu": 1.2, "xavierCpu": 0.6, "xavierGpu": 0.3, "pynqFpga": 2.0},
        "memoryRequirements": {"rpiCpu": 0.8, "xavierCpu": 0.5, "xavierGpu": 0.3, "pynqFpga": 0.4}
    }
}

def calculate_execution_time(task_type, client, server, static_conditions):
    """Calculate execution time for a single client-server-task combination"""
    
    # Base execution time (use best available platform)
    base_execution_time = min(task_type["executionTime"].values())
    
    # Network latency
    network_latency = static_conditions["network_latencies"].get((client, server), 0.01)
    
    # Queue time (average queue length * processing time per task)
    avg_queue_length = np.mean(list(static_conditions["queue_lengths"].values()))
    queue_time = avg_queue_length * 0.1  # Assume 0.1s per queued task
    
    # Cold start time (use best available platform)
    cold_start_time = min(task_type["coldStartDuration"].values())
    
    # I/O time (simplified)
    io_time = 0.05  # Simplified I/O overhead
    
    # Total execution time
    return base_execution_time + network_latency + queue_time + cold_start_time + io_time

# Calculate execution times for ALL client-server-task combinations
all_execution_times = {}

for task_type_name, task_type in task_types.items():
    for client in client_nodes:
        for server in server_nodes:
            if client != server:  # Skip self-execution
                execution_time = calculate_execution_time(task_type, client, server, static_conditions)
                key = (client, server, task_type_name)
                all_execution_times[key] = execution_time

print(f"Calculated execution times for {len(all_execution_times)} combinations")
print(f"Task types: {list(task_types.keys())}")
print(f"Total combinations: {len(task_types) * len(client_nodes) * len(server_nodes)}")

# Show some examples
print("\nExample execution times:")
for i, (key, time) in enumerate(list(all_execution_times.items())[:5]):
    client, server, task_type = key
    print(f"{client} -> {server} ({task_type}): {time:.3f}s")


Calculated execution times for 24 combinations
Task types: ['dnn1', 'dnn2']
Total combinations: 24

Example execution times:
client_node0 -> server_node0 (dnn1): 1.013s
client_node0 -> server_node1 (dnn1): 1.001s
client_node0 -> server_node2 (dnn1): 1.006s
client_node0 -> server_node3 (dnn1): 1.005s
client_node1 -> server_node0 (dnn1): 1.042s


## 3. Create Bipartite Graph for Batch Processing

Create a bipartite graph structure with clients, servers, and optionally tasks as nodes. This enables batch processing of multiple tasks and creates rich training data for GNNs.


In [ ]:
def create_bipartite_graph(client_nodes, server_nodes, task_batch, static_conditions, 
                          execution_times, include_task_nodes=True):
    """Create a bipartite graph for GNN training with clients, servers, and tasks"""
    
    # Node features
    node_features = {}
    
    # Client node features
    for client in client_nodes:
        node_features[client] = {
            "node_type": "client",
            "memory_usage": static_conditions["memory_usage"].get(client, 0.5),
            "queue_length": 0,  # Clients don't have execution queues
            "active_tasks": len([t for t in task_batch if t.get('source_client') == client])
        }
    
    # Server node features
    for server in server_nodes:
        node_features[server] = {
            "node_type": "server",
            "memory_usage": static_conditions["memory_usage"].get(server, 0.5),
            "queue_length": np.mean(list(static_conditions["queue_lengths"].values())),
            "platform_utilization": random.uniform(0.2, 0.8)  # Simulated utilization
        }
    
    # Task node features (optional - for edge prediction)
    task_nodes = []
    if include_task_nodes:
        for i, task in enumerate(task_batch):
            task_node_id = f"task_{i}"
            task_nodes.append(task_node_id)
            
            node_features[task_node_id] = {
                "node_type": "task",
                "task_type": task['task_type'],
                "source_client": task['source_client'],
                "memory_requirement": task.get('memory_requirement', 0.5),
                "priority": task.get('priority', 1.0)
            }
    
    # Edge features
    edge_features = {}
    
    # Client-Server edges (bipartite connections)
    for client in client_nodes:
        for server in server_nodes:
            if client != server:
                edge_key = (client, server)
                
                # Network latency
                network_latency = static_conditions["network_latencies"].get(edge_key, 0.01)
                
                # Execution times for all task types
                task_execution_times = {}
                for task_type in task_types.keys():
                    execution_time = execution_times.get((client, server, task_type), 0.0)
                    task_execution_times[task_type] = execution_time
                
                edge_features[edge_key] = {
                    "edge_type": "client_server",
                    "network_latency": network_latency,
                    "execution_times": task_execution_times
                }
    
    # Task-Client and Task-Server edges (if task nodes included)
    if include_task_nodes:
        for i, task in enumerate(task_batch):
            task_node_id = f"task_{i}"
            source_client = task['source_client']
            task_type = task['task_type']
            
            # Task originates from client (fixed edge)
            client_edge = (task_node_id, source_client)
            edge_features[client_edge] = {
                "edge_type": "task_origin",
                "weight": 1.0
            }
            
            # Task can be executed on compatible servers (prediction targets)
            for server in server_nodes:
                if server != source_client:  # Don't include self-execution
                    server_edge = (task_node_id, server)
                    execution_time = execution_times.get((source_client, server, task_type), 0.0)
                    
                    edge_features[server_edge] = {
                        "edge_type": "task_execution",
                        "execution_time": execution_time,
                        "predicted_performance": 1.0 / (execution_time + 0.001)  # Higher is better
                    }
    
    return {
        "client_nodes": client_nodes,
        "server_nodes": server_nodes,
        "task_nodes": task_nodes,
        "all_nodes": client_nodes + server_nodes + task_nodes,
        "node_features": node_features,
        "edge_features": edge_features,
        "graph_type": "bipartite",
        "batch_size": len(task_batch)
    }

# Create example task batch
task_batch = [
    {'task_type': 'dnn1', 'source_client': 'client_node0', 'memory_requirement': 0.3, 'priority': 1.0},
    {'task_type': 'dnn2', 'source_client': 'client_node1', 'memory_requirement': 0.5, 'priority': 2.0},
    {'task_type': 'dnn1', 'source_client': 'client_node2', 'memory_requirement': 0.4, 'priority': 1.5},
    {'task_type': 'dnn2', 'source_client': 'client_node0', 'memory_requirement': 0.6, 'priority': 1.8}
]

# Create bipartite graph
bipartite_graph = create_bipartite_graph(
    client_nodes=client_nodes,
    server_nodes=server_nodes,
    task_batch=task_batch,
    static_conditions=static_conditions,
    execution_times=all_execution_times,
    include_task_nodes=True
)

print(f"Bipartite graph created with:")
print(f"  - {len(bipartite_graph['client_nodes'])} client nodes")
print(f"  - {len(bipartite_graph['server_nodes'])} server nodes")
print(f"  - {len(bipartite_graph['task_nodes'])} task nodes")
print(f"  - {len(bipartite_graph['node_features'])} total nodes")
print(f"  - {len(bipartite_graph['edge_features'])} edges")
print(f"  - Batch size: {bipartite_graph['batch_size']}")


Bipartite graph created with:
  - 3 client nodes
  - 4 server nodes
  - 4 task nodes
  - 11 total nodes
  - 32 edges
  - Batch size: 4


## 4. Generate Multiple Training Samples with Batch Processing

Generate multiple training samples, each containing a batch of tasks with fully populated execution times.


In [ ]:
def generate_batch_training_samples(num_samples=5, batch_size=4):
    """Generate multiple training samples with batch processing"""
    
    training_samples = []
    task_type_names = list(task_types.keys())
    
    for sample_idx in range(num_samples):
        print(f"Generating sample {sample_idx + 1}/{num_samples}")
        
        # Create slightly different static conditions for each sample
        sample_static_conditions = static_conditions.copy()
        sample_static_conditions["queue_lengths"] = {
            k: v + random.randint(-1, 1) for k, v in static_conditions["queue_lengths"].items()
        }
        
        # Recalculate execution times for this sample
        sample_execution_times = {}
        for task_type_name, task_type in task_types.items():
            for client in client_nodes:
                for server in server_nodes:
                    if client != server:
                        execution_time = calculate_execution_time(
                            task_type, client, server, sample_static_conditions
                        )
                        key = (client, server, task_type_name)
                        sample_execution_times[key] = execution_time
        
        # Generate random task batch
        sample_task_batch = []
        for task_idx in range(batch_size):
            task = {
                'task_id': f"sample_{sample_idx}_task_{task_idx}",
                'task_type': random.choice(task_type_names),
                'source_client': random.choice(client_nodes),
                'memory_requirement': random.uniform(0.2, 0.8),
                'priority': random.uniform(0.5, 2.0),
                'arrival_time': task_idx * 0.1  # Staggered arrivals
            }
            sample_task_batch.append(task)
        
        # Create bipartite graph for this sample
        sample_graph = create_bipartite_graph(
            client_nodes=client_nodes,
            server_nodes=server_nodes,
            task_batch=sample_task_batch,
            static_conditions=sample_static_conditions,
            execution_times=sample_execution_times,
            include_task_nodes=True
        )
        
        # Simulate scheduling decisions (ground truth)
        scheduling_decisions = {}
        for i, task in enumerate(sample_task_batch):
            task_node_id = f"task_{i}"
            source_client = task['source_client']
            task_type = task['task_type']
            
            # Find best server (ground truth) - lowest execution time
            best_server = None
            best_time = float('inf')
            
            for server in server_nodes:
                if server != source_client:
                    exec_time = sample_execution_times.get((source_client, server, task_type), float('inf'))
                    if exec_time < best_time:
                        best_time = exec_time
                        best_server = server
            
            scheduling_decisions[task_node_id] = {
                'assigned_server': best_server,
                'execution_time': best_time,
                'source_client': source_client,
                'task_type': task_type
            }
        
        # Create training sample
        sample = {
            "sample_id": sample_idx,
            "bipartite_graph": sample_graph,
            "task_batch": sample_task_batch,
            "execution_times": sample_execution_times,
            "static_conditions": sample_static_conditions,
            "scheduling_decisions": scheduling_decisions,
            "batch_size": batch_size,
            "total_combinations": len(sample_execution_times)
        }
        
        training_samples.append(sample)
    
    return training_samples

# Generate training samples
training_samples = generate_batch_training_samples(num_samples=5, batch_size=4)

print(f"\nGenerated {len(training_samples)} training samples")
print(f"Each sample contains:")
print(f"  - {len(training_samples[0]['bipartite_graph']['all_nodes'])} nodes")
print(f"  - {len(training_samples[0]['bipartite_graph']['edge_features'])} edges")
print(f"  - {len(training_samples[0]['execution_times'])} execution time measurements")
print(f"  - {training_samples[0]['batch_size']} tasks per batch")
print(f"  - {len(training_samples[0]['scheduling_decisions'])} scheduling decisions")


Generating sample 1/5
Generating sample 2/5
Generating sample 3/5
Generating sample 4/5
Generating sample 5/5

Generated 5 training samples
Each sample contains:
  - 11 nodes
  - 32 edges
  - 24 execution time measurements
  - 4 tasks per batch
  - 4 scheduling decisions


## 5. Analyze and Save Training Data

Analyze the training data quality and save it in formats suitable for GNN training.


In [ ]:
# Analyze training data
def analyze_training_data(training_samples):
    """Analyze the quality and completeness of training data"""
    
    print("=== Training Data Analysis ===")
    print(f"Total samples: {len(training_samples)}")
    
    # Basic statistics
    total_tasks = sum(sample['batch_size'] for sample in training_samples)
    total_execution_times = sum(len(sample['execution_times']) for sample in training_samples)
    total_decisions = sum(len(sample['scheduling_decisions']) for sample in training_samples)
    
    print(f"Total tasks across all samples: {total_tasks}")
    print(f"Total execution time measurements: {total_execution_times}")
    print(f"Total scheduling decisions: {total_decisions}")
    
    # Graph structure analysis
    sample_graph = training_samples[0]['bipartite_graph']
    print(f"\n=== Graph Structure ===")
    print(f"Client nodes: {len(sample_graph['client_nodes'])}")
    print(f"Server nodes: {len(sample_graph['server_nodes'])}")
    print(f"Task nodes per sample: {len(sample_graph['task_nodes'])}")
    print(f"Total edges per sample: {len(sample_graph['edge_features'])}")
    
    # Edge type analysis
    edge_types = defaultdict(int)
    for edge_data in sample_graph['edge_features'].values():
        edge_types[edge_data['edge_type']] += 1
    
    print(f"\n=== Edge Types ===")
    for edge_type, count in edge_types.items():
        print(f"{edge_type}: {count} edges")
    
    # Execution time statistics
    all_execution_times = []
    for sample in training_samples:
        all_execution_times.extend(sample['execution_times'].values())
    
    print(f"\n=== Execution Time Statistics ===")
    print(f"Min execution time: {min(all_execution_times):.3f}s")
    print(f"Max execution time: {max(all_execution_times):.3f}s")
    print(f"Mean execution time: {np.mean(all_execution_times):.3f}s")
    print(f"Std execution time: {np.std(all_execution_times):.3f}s")
    
    return {
        'total_samples': len(training_samples),
        'total_tasks': total_tasks,
        'total_execution_times': total_execution_times,
        'execution_time_stats': {
            'min': min(all_execution_times),
            'max': max(all_execution_times),
            'mean': np.mean(all_execution_times),
            'std': np.std(all_execution_times)
        }
    }

# Save training data
def save_training_data(training_samples, output_dir):
    """Save training data in multiple formats"""
    
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    # 1. Save complete training samples as pickle
    with open(output_dir / "training_samples.pkl", "wb") as f:
        pickle.dump(training_samples, f)
    
    # 2. Save as JSON for inspection
    with open(output_dir / "training_samples.json", "w") as f:
        json.dump(training_samples, f, indent=2, default=str)
    
    # 3. Save metadata
    metadata = {
        "num_samples": len(training_samples),
        "client_nodes": client_nodes,
        "server_nodes": server_nodes,
        "task_types": list(task_types.keys()),
        "batch_size": training_samples[0]['batch_size'],
        "total_combinations_per_sample": len(training_samples[0]['execution_times']),
        "graph_structure": {
            "node_types": ["client", "server", "task"],
            "edge_types": ["client_server", "task_origin", "task_execution"],
            "bipartite": True
        }
    }
    
    with open(output_dir / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2, default=str)
    
    return output_dir

# Analyze and save the training data
analysis_results = analyze_training_data(training_samples)
output_dir = save_training_data(training_samples, "training_data_output")

print(f"\nTraining data saved to {output_dir}")
print(f"Files created:")
for file_path in output_dir.glob("*"):
    if file_path.is_file():
        print(f"  - {file_path.name}")


=== Training Data Analysis ===
Total samples: 5
Total tasks across all samples: 20
Total execution time measurements: 120
Total scheduling decisions: 20

=== Graph Structure ===
Client nodes: 3
Server nodes: 4
Task nodes per sample: 4
Total edges per sample: 32

=== Edge Types ===
client_server: 12 edges
task_origin: 4 edges
task_execution: 16 edges

=== Execution Time Statistics ===
Min execution time: 0.968s
Max execution time: 1.509s
Mean execution time: 1.230s
Std execution time: 0.203s


TypeError: keys must be str, int, float, bool or None, not tuple

## Summary: Training Data Generation for GNN Scheduling

This notebook demonstrates a comprehensive approach to generating training data for GNN-based scheduling with the following key innovations:

### ✅ **What We Achieved**

1. **Fully Populated Graphs**: Each training sample contains execution times for ALL possible client-server pairs, not just the one that was executed.

2. **Bipartite Graph Structure**: Created graphs with three types of nodes (clients, servers, tasks) and three types of edges (client-server, task-origin, task-execution).

3. **Batch Processing**: Demonstrated how to process multiple tasks simultaneously, creating richer training data.

4. **Static Conditions**: Made dynamic variables (queue lengths, memory usage, network latencies) static for consistent training data.

### 🎯 **Key Benefits for GNN Training**

- **Complete Information**: No missing data - every possible scheduling decision has performance data
- **Edge Prediction**: GNN can learn to predict optimal task→server edges
- **Batch Efficiency**: Process multiple tasks simultaneously for better training efficiency
- **Realistic Scenarios**: Includes network latencies, queue states, and resource constraints

### 🚀 **Next Steps**

1. **Integrate with Herosim**: Use the `BatchGNNScheduler` in actual simulations
2. **Scale Up**: Generate larger datasets with more nodes and tasks
3. **GNN Architecture**: Design GNN models that can handle bipartite graphs with heterogeneous nodes
4. **Training Pipeline**: Create training scripts that use this data format

### 📊 **Data Structure Summary**

Each training sample provides:
- **Bipartite Graph**: Clients ↔ Servers ↔ Tasks
- **Complete Execution Times**: All client-server-task combinations  
- **Ground Truth**: Actual scheduling decisions
- **Rich Features**: Node and edge attributes for learning

This approach transforms sparse simulation data (one execution per task) into dense training data (all possible executions), making it much more valuable for training GNNs that need to understand the complete system topology and make globally optimal scheduling decisions.


# Training Data Generation with Fully Populated Graphs

This notebook demonstrates how to generate training data with fully populated graphs for GNN scheduling, where each sample contains execution times for all possible client-server pairs.


In [ ]:
import sys
sys.path.append('../..')

import json
import pickle
from pathlib import Path
from typing import Dict, List, Tuple, Any
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.generator.traces import generate_exhaustive_training_workload
from src.placement.model import SimulationData, TimeSeries


## 1. Define Static Conditions

First, we create static conditions to make dynamic variables constant across all training samples.


In [2]:
def create_static_conditions_example():
    """Create example static conditions for training data generation"""
    
    # Define client and server nodes
    client_nodes = ["client_node0", "client_node1", "client_node2"]
    server_nodes = ["server_node0", "server_node1", "server_node2", "server_node3"]
    
    # Static queue lengths for platforms (platform_id -> queue_length)
    queue_lengths = {
        0: 2,   # Platform 0 has 2 tasks in queue
        1: 5,   # Platform 1 has 5 tasks in queue
        2: 0,   # Platform 2 has empty queue
        3: 3,   # Platform 3 has 3 tasks in queue
        4: 1,   # Platform 4 has 1 task in queue
        5: 4    # Platform 5 has 4 tasks in queue
    }
    
    # Static memory usage for nodes (node_name -> memory_usage_percentage)
    memory_usage = {
        "client_node0": 0.3,  # 30% memory usage
        "client_node1": 0.5,  # 50% memory usage
        "client_node2": 0.2,  # 20% memory usage
        "server_node0": 0.7,  # 70% memory usage
        "server_node1": 0.4,  # 40% memory usage
        "server_node2": 0.6,  # 60% memory usage
        "server_node3": 0.8   # 80% memory usage
    }
    
    # Static storage usage (storage_id -> bytes_used)
    storage_usage = {
        0: 5000000,   # 5MB used
        1: 2000000,   # 2MB used
        2: 8000000,   # 8MB used
        3: 3000000    # 3MB used
    }
    
    # Static network latencies for all client-server pairs
    network_latencies = {}
    for client in client_nodes:
        for server in server_nodes:
            if client != server:
                # Generate realistic network latencies
                if client.startswith("client_node0"):
                    # Client 0 has good connectivity
                    latency = random.uniform(0.001, 0.02)
                elif client.startswith("client_node1"):
                    # Client 1 has medium connectivity
                    latency = random.uniform(0.02, 0.05)
                else:
                    # Client 2 has poor connectivity
                    latency = random.uniform(0.05, 0.1)
                
                network_latencies[(client, server)] = latency
    
    return {
        "queue_lengths": queue_lengths,
        "memory_usage": memory_usage,
        "storage_usage": storage_usage,
        "network_latencies": network_latencies
    }, client_nodes, server_nodes

# Create static conditions
static_conditions, client_nodes, server_nodes = create_static_conditions_example()

print(f"Client nodes: {client_nodes}")
print(f"Server nodes: {server_nodes}")
print(f"Total client-server pairs: {len(client_nodes) * len(server_nodes)}")
print(f"Network latencies: {len(static_conditions['network_latencies'])} pairs")


Client nodes: ['client_node0', 'client_node1', 'client_node2']
Server nodes: ['server_node0', 'server_node1', 'server_node2', 'server_node3']
Total client-server pairs: 12
Network latencies: 12 pairs
